# แบบฝึกหัด 6: Model Deployment

**ชุดข้อมูล:** AI Student Impact Dataset 

**สมาชิก**\
6810405950 สิปปกร จันทร์พุ่ม \
6810405879 ศิรวิชญ์ กู้สกุลไพบูลย์


**หมู่เรียน:** 1

Dataset: https://www.kaggle.com/datasets/laveshjadon/ai-impact-on-students

---

## จุดประสงค์

สร้างโมเดลทำนายว่านักศึกษามีแนวโน้มที่ GPA หลังจบภาคเรียนจะดีขึ้นหรือไม่ จากข้อมูลพฤติกรรมการเรียนและการใช้ Generative AI เช่น ชั่วโมงการใช้ AI ต่อสัปดาห์ รูปแบบการใช้ AI การเข้าเรียน ระยะเวลาการนอน และ GPA ก่อนเริ่มภาคเรียน

โจทย์ที่เลือกใช้เป็นหลักคือ
> **นักศึกษามี GPA หลังจบภาคเรียนดีขึ้นจากก่อนเริ่มภาคเรียนหรือไม่**

โดยจะสร้าง label ชื่อ `GPA_Improved` จากค่า GPA ก่อนและหลังภาคเรียน


## วิดีโอสาธิตการทำนายของโมเดล
https://youtu.be/hHJYk5pSyIM

## 1) ระบบและ pipeline ที่ทำงานจริง

ระบบรับข้อมูล 7 features จากนักศึกษา แล้วส่งเป็น `DataFrame` เข้า **pipeline เดียวกับตอน train**:

1. `RobustScaler` แปลง features ตัวเลข
2. `OneHotEncoder(handle_unknown="ignore")` แปลง Primary_Use_Case
3. `OrdinalEncoder` แปลง Year_of_Study และ Prompt_Engineering_Skill
4. `RandomForestClassifier` ทำนาย class และ `predict_proba`
5. UI แสดงผล ความน่าจะเป็น ระดับความมั่นใจ และ guardrail

เลือก Random Forest เพราะ Validation Macro F1 สูงที่สุดใน HW5 จากนั้น refit ด้วย train+validation และรายงาน test

## 2) Export fitted pipeline และ metadata

ใช้ `joblib` บันทึก fitted pipeline และบันทึก metadata แยกเป็น JSON เพื่อให้ UI รู้ข้อมูลสำคัญๆ เช่น feature order, model version, metrics และ confidence threshold

โค้ดจริงอยู่ใน `train_model.py`

In [15]:
from train_model import train_and_export

deployment_metadata = train_and_export()

print("Selected model:", deployment_metadata["model_name"])
print("Test Accuracy:", round(deployment_metadata["test_metrics"]["accuracy"], 4))
print("Test Macro F1:", round(deployment_metadata["test_metrics"]["macro_f1"], 4))
print("Saved: pipeline_student_impact.joblib")
print("Saved: model_metadata.json")


Selected model: Random Forest
Test Accuracy: 0.7741
Test Macro F1: 0.6389
Saved: pipeline_student_impact.joblib
Saved: model_metadata.json


## 3) การออกแบบ UI

**Target User:** นักศึกษาที่ต้องการทดลองประเมินแนวโน้มว่า GPA จะดีขึ้นหรือไม่จากพฤติกรรมการเรียนและการใช้ Generative AI

ระบบพัฒนาด้วย **Streamlit** โดยออกแบบ UI ให้เรียบง่ายและใช้งานได้โดยไม่จำเป็นต้องมีความรู้ด้าน Machine Learning

ผู้ใช้กรอกข้อมูลทั้งหมด 7 features ผ่าน input ได้แก่
- `number_input` สำหรับจำนวนชั่วโมงอ่านหนังสือและจำนวนชั่วโมงใช้ Generative AI ต่อสัปดาห์
- `selectbox` สำหรับชั้นปี วัตถุประสงค์หลักในการใช้ AI และระดับทักษะ Prompt Engineering
- `slider` สำหรับระดับการพึ่งพา AI และระดับความกังวลระหว่างสอบ

เมื่อกดปุ่ม **Predict** ระบบจะแสดง
- ผลการทำนายว่า GPA มีแนวโน้มดีขึ้นหรือไม่
- Confidence ของโมเดล
- Probability ของแต่ละ class
- Prediction latency
- คำเตือนเมื่อ confidence ต่ำกว่าเกณฑ์ หรือข้อมูลอยู่นอกช่วงที่พบในชุด train

## 4) สิ่งที่ต้องเตรียมก่อนรันระบบ

ไฟล์ที่จำเป็นสำหรับการรันระบบประกอบด้วย

```text
project/
├── app.py
├── pipeline_student_impact.joblib
├── model_metadata.json
└── requirements.txt
```

โดย

- `app.py` เป็น Streamlit application สำหรับรับ input และแสดงผล
- `pipeline_student_impact.joblib` เป็น pipeline และโมเดลที่ train เสร็จแล้ว
- `model_metadata.json` เก็บข้อมูลที่ระบบต้องใช้ เช่น categories, confidence threshold
- `requirements.txt` ระบุ package ที่จำเป็น

ตัวอย่าง `requirements.txt`
```text
streamlit
pandas
scikit-learn
joblib
```

ติดตั้ง package ด้วย
```bash
pip install -r requirements.txt
```

จากนั้นรันด้วย
```bash
python -m streamlit run app.py
```

Streamlit ใช้ port `8501` เป็นค่าเริ่มต้น และสามารถเข้าใช้งานผ่าน
```text
http://localhost:8501
```
ระบบไม่จำเป็นต้อง train โมเดลใหม่ทุกครั้ง เนื่องจาก `app.py` จะโหลดโมเดลที่บันทึกไว้ใน `pipeline_student_impact.joblib` และ metadata จาก `model_metadata.json` มาใช้ในการทำนายโดยตรง

## 5) ข้อจำกัดของระบบ

### ข้อจำกัดของ Input
โมเดลรองรับข้อมูลทั้งหมด 7 features ตามรูปแบบที่ใช้ตอน train เท่านั้น
ข้อมูลประเภท category สามารถเลือกได้เฉพาะค่าที่กำหนดไว้ใน `model_metadata.json` ส่วนข้อมูลประเภทตัวเลขจะถูกตรวจสอบกับช่วงข้อมูลที่พบในชุด train
หากค่าตัวเลขอยู่นอกช่วงที่โมเดลเคยเรียนรู้ ระบบยังสามารถทำนายได้ แต่จะแสดงคำเตือนว่าข้อมูลอยู่นอก training range และผลลัพธ์อาจมีความน่าเชื่อถือลดลง

### กรณีที่ทำนายผิดพลาด
ผลลัพธ์เป็นการทำนายจากรูปแบบที่โมเดลเรียนรู้จากชุดข้อมูล จึงไม่สามารถรับประกันได้ว่าจะทำนายถูกต้องทุกกรณี
โมเดลอาจผิดพลาดได้เมื่อ
- input แตกต่างจากข้อมูลที่ใช้ train มาก
- input อยู่นอกช่วงที่โมเดลเคยพบ
- probability ของทั้งสอง class ใกล้เคียงกัน

### Confidence Threshold
ระบบคำนวณ confidence จาก probability สูงสุดที่ได้จาก `predict_proba()` และกำหนด **confidence threshold เท่ากับ 65%**
หาก
```text
confidence >= 65%
```
ระบบจะแสดงผลการทำนายตามปกติ
แต่หาก
```text
confidence < 65%
```
ระบบจะแสดงคำเตือน **Low confidence** เพื่อบอกว่าผลการทำนายของโมเดลยังไม่ชัดเจน
ตัวอย่าง
```text
GPA Improved:     78%
GPA Not Improved: 22%
Confidence = 78%
→ ทำนายตามปกติ
```

แต่หาก
```text
GPA Improved:     53%
GPA Not Improved: 47%
Confidence = 53%
→ Low confidence
```

---
# ส่วนท้าย

## Deployment ที่ใช้งานจริง

- Streamlit: https://ai-student-impact-tar-jpg.streamlit.app/
- GitHub: https://github.com/Tar-jpg/ML-Assignment
- Runtime: Python 3.11

## การมีส่วนร่วมของสมาชิกแต่ละคนในกลุ่ม

- นายสิปปกร จันทร์พุ่ม — ตรวจ pipeline/model, monitoring และทดสอบ deployment
- นายศิรวิชญ์ กู้สกุลไพบูลย์ — ออกแบบ Streamlit UI, demo cases และทำสคริปต์อธิบาย

## การเปิดเผยการใช้เครื่องมือปัญญาประดิษฐ์

**เครื่องมือที่ใช้:** ChatGPT/Codex

**ใช้เพื่อ:** ช่วยอ่านโจทย์และเอกสารประกอบ วางโครงสร้าง deployment เขียน Streamlit UI สร้าง monitoring/continual-learning plan และช่วยตรวจความครบถ้วนของงาน

**การตรวจสอบโดยผู้จัดทำ:** ตรวจ feature/label/pipeline กับ HW5 รัน train และ prediction จริง ตรวจ model metrics ตรวจ evironment
